# GISTS

 0. LoadData+SampleSelection_sentiment140_tweets.py 

In [5]:
import pandas as pd

#create input and output directories
import os
inputpath = 'input'
outputpath = 'outputs'
if os.path.exists(inputpath) is False:
    os.mkdir(inputpath)
if os.path.exists(outputpath) is False:
    os.mkdir(outputpath)
    
#input file path
sentiment140_file = 'input/training.1600000.processed.noemoticon.csv'
   
#read csv file
colnames = ['polarity', 'id', 'post_datetime', 'query', 'user', 'tweet']
df_tweets = pd.read_csv(sentiment140_file,
                encoding='UTF', names=colnames, encoding_errors='ignore')

# get 1600 tweets
df = df_tweets[['polarity','tweet']].sample(n=1600, random_state=0)
df.to_csv("outputs/selected_tweets1600.csv", index=False)

In [2]:
df_tweets.polarity.value_counts()

0    800000
4    800000
Name: polarity, dtype: int64

In [3]:
df.polarity.value_counts()

0    811
4    789
Name: polarity, dtype: int64

1. text-cleaning.py

In [4]:
import re
from autocorrect import Speller
spell = Speller(lang='en')
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
lemm = WordNetLemmatizer()

#Fixing Word Lengthening
def reduce_lengthening(text):
    pattern = re.compile(r"(.)\1{2,}")
    return pattern.sub(r"\1\1", text)

def text_preprocess(doc):
    #Lowercasing all the letters
    temp = doc.lower()
    #Removing hashtags and mentions
    temp = re.sub("@[A-Za-z0-9_]+","", temp)
    temp = re.sub("#[A-Za-z0-9_]+","", temp)
    #Removing links
    temp = re.sub(r"http\S+", "", temp)
    temp = re.sub(r"www.\S+", "", temp)
    #removing numbers
    temp = re.sub("[0-9]","", temp)
    #Removing '
    temp = re.sub("'"," ",temp)

    #Tokenization
    temp = word_tokenize(temp)
    #Fixing Word Lengthening
    temp = [reduce_lengthening(w) for w in temp]
    #spell corrector
    temp = [spell(w) for w in temp]
    #stem
    temp = [lemm.lemmatize(w) for w in temp]
    #Removing short words
    temp = [w for w in temp if len(w)>2]
    temp = " ".join(w for w in temp)
    
    return temp

In [5]:
text_preprocess("I LOOOOVE it!!!")

'love'

 2. train_test_split.py

In [6]:
from sklearn.model_selection import train_test_split
#replacing textual categories by integers
X = df.tweet
y = df.polarity.replace(4,1)
class_names = ['negative', 'positive']

#split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

print("training size:", len(X_train))
print("testing size:", len(X_test))

training size: 1120
testing size: 480


3. NaiveBayes-SentimentAnalysis.py

In [9]:
#naive bayes libraries
from nltk import FreqDist
from nltk.classify import NaiveBayesClassifier,accuracy

#build the dataset
def build_dataset(X,y):
    #build the dataset
    words = [text_preprocess(word).split(" ") for word in X]
    dataset = list(zip(words, y))
    
    return dataset

# Define the feature extractor with the 2000 most used words
all_words = FreqDist(sum([w.split(" ") for w in X_train],[]))
word_features = list(all_words)[:2000]
    
def document_features(words):
    features = {}
    for word in word_features:
        features['contains({})'.format(word)] = (word in set(words))

    return features

trainset = build_dataset(X_train, y_train)
testset = build_dataset(X_test, y_test)
train_set = [(document_features(d), y) for (d,y) in trainset]
test_set = [(document_features(d), y) for (d,y) in testset]

#training
nb_classifier = NaiveBayesClassifier.train(train_set)

# Test the classifier
print('Test score: %.2f\n' % accuracy(nb_classifier, test_set))

Test score: 0.69



In [10]:
# Show the most important features as interpreted by Naive Bayes
nb_classifier.show_most_informative_features(5)

Most Informative Features
          contains(look) = True                1 : 0      =      6.5 : 1.0
          contains(miss) = True                0 : 1      =      6.2 : 1.0
         contains(those) = True                1 : 0      =      5.9 : 1.0
        contains(thanks) = True                1 : 0      =      5.4 : 1.0
          contains(dont) = True                0 : 1      =      5.2 : 1.0


4. catboost-SentimentAnalysis.py

In [11]:
from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer
from sklearn.pipeline import make_pipeline
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

#data preparation
cleaned_train = [text_preprocess(d) for d in X_train]
cleaned_test = [text_preprocess(d) for d in X_test]

pipe = make_pipeline(CountVectorizer(), TfidfTransformer())
pipe.fit(cleaned_train)

Xtrain = pipe.transform(cleaned_train)
Xtest = pipe.transform(cleaned_test)

#training
cat_classifier = CatBoostClassifier(iterations=300,
                         learning_rate=0.001,
                         depth=8,
                         eval_metric='CrossEntropy',
                         od_type='Iter',
                         od_wait=30,
                         silent=True
                        )
cat_classifier.fit(Xtrain, y_train,eval_set=(Xtest, y_test),plot=True)

#prediction
predictions = cat_classifier.predict(Xtest)
print('Test score: %.2f\n' % (accuracy_score(y_test, predictions)))

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Test score: 0.68



5. Fasttextembedding+LSTM-SentimentAnalysis.py

In [12]:
import numpy as np
import tensorflow as tf
import keras
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential,Model
from keras.layers.embeddings import Embedding
from keras.layers import Dense,Dropout,Bidirectional,LSTM

#################data preparation
cleaned_train = [text_preprocess(d) for d in X_train]
#building a dictionary
tk = Tokenizer(num_words=None) #the maximum number of words to keep, based on word frequency
tk.fit_on_texts(cleaned_train)
#1.1 get the size of the dictionary
dico_size = len(tk.word_counts.items())
num_tokens = dico_size + 1
#2. building a sequneces
seq_X = tk.texts_to_sequences(cleaned_train)
#2.1 calculate maxi length of tweets
max_len = np.max(np.array([len(d) for d in seq_X]))
marg_len=10
maxlen = max_len + marg_len
#3. padding the sequences
Xtrain = pad_sequences(seq_X,maxlen=maxlen,padding='post' )


###########loading glove and fasttext embedding vectors
def load_embedding_model(file):

    embedding_model = {}
    with open(file,'r') as f:
        for line in f:
            split_line = line.split()
            word = split_line[0]
            embedding = np.array(split_line[1:], dtype=np.float64)
            embedding_model[word] = embedding
    return embedding_model

embedding_index_fasttext = load_embedding_model('input/wiki-news-300d-1M.vec')
print('found %s word vectors in loaded fasttext model.' % len(embedding_index_fasttext))


##########Preparing a corresponding embedding matrix
def embedding_matrix(num_tokens,embedding_dim,embedding_index):
    hits=0
    misses=[]

    # Prepare embedding matrix
    embedding_matrix = np.zeros((num_tokens, embedding_dim))

    for word, i in tk.word_index.items():
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            # Words not found in embedding index will be all-zeros.
            # This includes the representation for "padding" and "OOV"
            embedding_matrix[i] = embedding_vector
            hits += 1
        else:
            misses.append(word)
    print("Converted %d words (%d misses)" % (hits, len(misses)))
    print("words not included in pretrained model:",misses)
    return embedding_matrix

embedding_dim=300
embedding_matrix_fasttext = embedding_matrix(num_tokens,embedding_dim,embedding_index_fasttext)



found 999995 word vectors in loaded fasttext model.
Converted 2579 words (74 misses)
words not included in pretrained model: ['loveyouu', 'lmaz', 'gottabesomebody', 'auctionsniper', 'funcionou', 'ncis', 'spartak', 'twitterberry', 'copiedandpasted', 'omgosh', 'catal', 'cookiedough', 'tagaytay', 'arrgghh', 'krystinas', 'kutnerr', 'camila', 'obnoxciously', 'rushmore', 'honeytint', 'cahntilli', 'fuckingtastic', 'allyssas', 'huhuhu', '…needed', 'bassotti', 'citrixcloud', 'wolfmother', 'kirstie', 'cheol', 'vilmarie', 'xaviermedia', 'heartburny', 'pnas', 'doliviawilder', 'techhelp', 'grimshaw', 'gyokoro', 'reblipping', 'followfriday', 'mbb', 'organization�', 'twitpics', 'lastlatter', 'pawngame', 'arangurens', 'konstantino', 'goodmorning', 'beeteedubs', 'pahonorsocietyst', 'farewellness', 'bellarlly', 'saymyspacetwitters', 'gottwitter', 'wwd', 'fianc�', 'triginometery', 'atikah', 'ilovejb', 'lismore', 'bodypump', 'groundctrl', 'lemmiin', 'tysonritteraar', 'anacecii', 'btnreply', 'wesseltof', '

In [34]:
##########Building the model with the embedding layer non trainable
embedding_layer_fasttext = Embedding(
    input_dim=num_tokens,
    output_dim=embedding_dim,
    input_length=maxlen,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix_fasttext),
    trainable=False,
)
fasttext_model = Sequential()
#enbedding
fasttext_model.add(embedding_layer_fasttext)
fasttext_model.add(Dropout(0.5))
#LSTM
fasttext_model.add(Bidirectional(LSTM(8,dropout=0.5,recurrent_dropout=0.2)))
# add a vanilla hidden layer:
fasttext_model.add(Dense(64, activation='relu'))
fasttext_model.add(Dense(16, activation='relu'))
fasttext_model.add(Dropout(0.5))
fasttext_model.add(Dense(units=1, activation='sigmoid',name='predictions'))

#compiling the model
fasttext_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-3, epsilon=1e-08, clipnorm=1.0), 
              loss="binary_crossentropy",
              metrics=['accuracy'])

print(fasttext_model.summary())

###########Trainning and evoluating the model
history = fasttext_model.fit(Xtrain, y_train, 
                            batch_size=parameterization.get('batch_size'), 
                            epochs=NUM_EPOCHS,
                             validation_split=0.2, verbose=2)
#test
cleaned_test = [text_preprocess(d) for d in X_test]
Qtest = tk.texts_to_sequences(cleaned_test)
Ptest = pad_sequences(Qtest,maxlen=maxlen,padding='post' )
print("LSTM model evaluation with fasttext 300d model:")
print(fasttext_model.evaluate(Ptest, y_test))


Model: "sequential_13"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_13 (Embedding)    (None, 37, 300)           796200    
                                                                 
 dropout_24 (Dropout)        (None, 37, 300)           0         
                                                                 
 bidirectional_14 (Bidirecti  (None, 16)               19776     
 onal)                                                           
                                                                 
 dense_24 (Dense)            (None, 64)                1088      
                                                                 
 dense_25 (Dense)            (None, 16)                1040      
                                                                 
 dropout_25 (Dropout)        (None, 16)                0         
                                                     

6. BERT-SentimentAnalysis.py

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
# Hide GPU from visible devices
tf.config.set_visible_devices([], 'GPU')
from transformers import BertTokenizer, TFBertForSequenceClassification
#load the model
bert_model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased")
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

#################text cleaning
def preprocess(X):
    import re
    def text_clean(text):
        temp = text.lower()
        temp = re.sub("@[A-Za-z0-9_]+","", temp)
        temp = re.sub("#[A-Za-z0-9_]+","", temp)
        temp = re.sub(r"http\S+", "", temp)
        temp = re.sub(r"www.\S+", "", temp)
        temp = re.sub("[0-9]","", temp)
        return temp
    X_cleaned = [text_clean(text) for text in X]
    return X_cleaned
############transforming raw data to an appropriate format ready to feed into the BERT model
def convert_example_to_feature(text):
    return bert_tokenizer.encode_plus(text,
            add_special_tokens = True, # add [CLS], [SEP]
            max_length = 128, # max length of the text that can go to BERT
            pad_to_max_length = True, # add [PAD] tokens
            return_attention_mask = True, # add attention mask to not focus on pad tokens
          )
def map_example_to_dict(input_ids, attention_masks, token_type_ids, label):
    return {
      "input_ids": input_ids,
      "token_type_ids": token_type_ids,
      "attention_mask": attention_masks,
    }, label
def encode_examples(X,y):
    input_ids_list = []
    token_type_ids_list = []
    attention_mask_list = []
    label_list = []
    for text, label in zip(X, y):
        bert_input = convert_example_to_feature(text)
        input_ids_list.append(bert_input['input_ids'])
        token_type_ids_list.append(bert_input['token_type_ids'])
        attention_mask_list.append(bert_input['attention_mask'])
        label_list.append([label])
    return tf.data.Dataset.from_tensor_slices((input_ids_list, attention_mask_list, token_type_ids_list, label_list)).map(map_example_to_dict)

# train dataset
ds_train_encoded = encode_examples(preprocess(X_train), y_train).shuffle(100).batch(32)
# test dataset
ds_test_encoded = encode_examples(preprocess(X_test), y_test).batch(32)

######### compiling the model
learning_rate = 3e-5
# choosing Adam optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, epsilon=1e-08)
# we do not have one-hot vectors, we can use sparce categorical cross entropy and accuracy
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')
bert_model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

#############training and evaluating
bert_model.fit(ds_train_encoded, epochs=2, validation_data=ds_test_encoded)

loss, acc = bert_model.evaluate(ds_test_encoded, verbose=0)
print("accuracy: {:5.2f}%".format(100 * acc))

##################Saving the model
bert_model.save_pretrained("outputs/bert_model", saved_model=True)